In [3]:
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.optim import Adam
from torchvision import datasets, transforms


In [4]:
print('==> Preparing data.............................')


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.datasets import MNIST, CIFAR10, CIFAR100, SVHN

import os
import torch
from torch import nn,optim
import torch.nn.functional as F

from torchvision import datasets, transforms

from time import perf_counter

import  numpy as np
import torch.utils.data as Data

from torch.utils.data import Dataset, DataLoader

class Safeman(Dataset):
    
    def __init__(self, data,targets):
        super(Safeman, self).__init__()
        self.data = data
        self.targets = targets
        
     
    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        img, target = self.data[idx], self.targets[idx]
        return img, target

class Safeman_Filter(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        mask, new_targets = [], []
        for i in range(len(targets)):
            if targets[i] in known:
                mask.append(i)
                new_targets.append(known.index(targets[i]))
        self.targets = np.array(new_targets)
        mask = torch.tensor(mask).long()
        self.data = torch.index_select(self.data, 0, mask)
        
class Safeman_FilterB(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        new_targets = []
        for i in range(len(targets)):
            if targets[i] in known:
                new_targets.append(0)
            else:
                new_targets.append(1)
        self.targets = np.array(new_targets)
        self.data = self.data

class Safeman_FilterC(Safeman):
    
    def __Filter__(self, trainknown):
        train_class_num=len(trainknown)
        for i in range(0,len(self.targets)) :
            if self.targets[i]>train_class_num:
                self.targets[i] = train_class_num
        self.data = self.data

        
        
class Safeman_FilterF(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        mask, new_targets = [], []
        for i in range(len(targets)):
            if targets[i] in known:
                mask.append(i)
                dd = known.index(targets[i])
                if dd == 4:
                    new_targets.append(0)
                else:
                    new_targets.append(1)                   
                    
                #new_targets.append(known.index(targets[i]))
        self.targets = np.array(new_targets)
        mask = torch.tensor(mask).long()
        self.data = torch.index_select(self.data, 0, mask)   


        
def setup_seed(seed):

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

setup_seed(8)



known=[0, 1, 2,3,4,5,6,7]
unknown=[ 5,6,7]





X_train0 = np.load('./TONdataset/x_train_iot1028+1del.npy')
y_train1 = np.load('./TONdataset/y_train_iot1028+1del.npy')
X_final_test0 = np.load('./TONdataset/x_test_iot1028+1del.npy' )
y__final_test1 = np.load('./TONdataset/y_test_iot1028+1del.npy')





X_train1=[]
X_final_test1=[]

for i in range(len(y_train1)):
    a = np.resize(X_train0[i], (1, 28, 28))
    X_train1 += [a]
    
for j in range(len(y__final_test1)):
    b = np.resize(X_final_test0[j], (1, 28, 28))
    X_final_test1 += [b]

i=0
j=0



x_train, x_test, y_train,y_test = torch.Tensor(X_train1), torch.Tensor(X_final_test1), torch.from_numpy(y_train1), torch.from_numpy(y__final_test1)

print(x_train.shape, x_test.shape, y_train.shape,y_test.shape)

train_dataset = Data.TensorDataset(x_train, y_train)
train_dataset.data = train_dataset.tensors[0]
train_dataset.targets = train_dataset.tensors[1]



test_dataset = Data.TensorDataset(x_test, y_test)
test_dataset.data = test_dataset.tensors[0]
test_dataset.targets = test_dataset.tensors[1]


labels =['backdoor', 'ddos', 'dos', 'injection', 'normal', 'password', 'scanning', 'xss']

train_dataset.classes = labels
test_dataset.classes = labels

train_dataset.classes_to_idx = {i: label for i, label in enumerate(labels)}
test_dataset.classes_to_idx = {i: label for i, label in enumerate(labels)}

num_class=len(labels)

b_s=256




trainset = Safeman_Filter(data=train_dataset.data,targets=train_dataset.targets)
print('All down Train Data:', len(trainset))
trainset.__Filter__(known=known)

#0930
train_loader = torch.utils.data.DataLoader(
    trainset, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)


print('Real train Data:', len(trainset))



testsetA = Safeman_Filter(data=test_dataset.data,targets=test_dataset.targets)
print('All testsetA Data:', len(testsetA))
testsetA.__Filter__(known=known)


test_loader_A = torch.utils.data.DataLoader(
    testsetA, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)


print('Real testsetA Data:', len(testsetA))


print("done!")

==> Preparing data.............................
torch.Size([242000, 1, 28, 28]) torch.Size([48000, 1, 28, 28]) torch.Size([242000]) torch.Size([48000])
All down Train Data: 242000
Real train Data: 242000
All testsetA Data: 48000
Real testsetA Data: 48000
done!


In [5]:
unique,counts = np.unique(trainset.targets,return_counts=True)
print(unique, counts)

[0 1 2 3 4 5 6 7] [ 16000  16000  16000  16000 130000  16000  16000  16000]


In [6]:
X_trainset_data=trainset.data
X_trainset_targets=trainset.targets

In [7]:
unique,counts = np.unique(X_trainset_targets,return_counts=True)
print(unique, counts)

[0 1 2 3 4 5 6 7] [ 16000  16000  16000  16000 130000  16000  16000  16000]


In [8]:
X_trainset_targets

array([6, 6, 6, ..., 0, 0, 0])

In [9]:
count=[10, 10, 10, 10, 13000, 10, 10, 10]
num_class=8
lists = [[] for i in range(num_class)]
y_train_temp=[]
x_train_temp=[]

In [10]:
for i in range(len(X_trainset_targets)):
    if len(lists[X_trainset_targets[i]])<count[X_trainset_targets[i]]:
        lists[X_trainset_targets[i]].append(X_trainset_targets[i])   
        y_train_temp+=[X_trainset_targets[i]]
        a = np.resize(X_trainset_data[i], (1, 64, 64))
        x_train_temp += [a]


In [11]:
unique,counts = np.unique(y_train_temp,return_counts=True)
print(unique, counts)

[0 1 2 3 4 5 6 7] [   10    10    10    10 13000    10    10    10]


In [12]:
x_train_temp[1]

array([[[0.0000000e+00, 4.0047044e-01, 6.5473884e-01, ...,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00],
        [4.0047044e-01, 6.5473884e-01, 3.0163100e-01, ...,
         0.0000000e+00, 0.0000000e+00, 4.0047044e-01],
        [6.5473884e-01, 3.0163100e-01, 2.5041966e-02, ...,
         0.0000000e+00, 4.0047044e-01, 6.5473884e-01],
        ...,
        [9.2891190e-07, 0.0000000e+00, 0.0000000e+00, ...,
         3.0163100e-01, 2.5041966e-02, 5.0000000e-01],
        [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ...,
         2.5041966e-02, 5.0000000e-01, 0.0000000e+00],
        [0.0000000e+00, 0.0000000e+00, 5.0000000e-01, ...,
         5.0000000e-01, 0.0000000e+00, 0.0000000e+00]]], dtype=float32)

In [13]:
x_train2, y_train2 = torch.Tensor(x_train_temp), torch.Tensor(y_train_temp)

print(x_train2.shape, y_train2.shape)

train_dataset2 = Data.TensorDataset(x_train2, y_train2)
train_dataset2.data = train_dataset2.tensors[0]
train_dataset2.targets = train_dataset2.tensors[1]


trainset2 = Safeman_Filter(data=train_dataset2.data,targets=train_dataset2.targets)
print('All down Train Data:', len(trainset2))
trainset2.__Filter__(known=known)



b_s=1

train_loader2 = torch.utils.data.DataLoader(
    trainset2, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)


print('Real train Data2:', len(trainset2))

torch.Size([13070, 1, 64, 64]) torch.Size([13070])
All down Train Data: 13070
Real train Data2: 13070


In [14]:
import random
import torch
import torch.nn as nn
from torchvision.utils import make_grid
import torch.optim as optim
import numpy as np
import torch.utils.data
import torchvision
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import torchvision.utils as vutils

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [15]:
class Generator(nn.Module):

    def __init__(self):
        super(Generator, self).__init__()

        # input 100*1*1
        self.layer1 = nn.Sequential(nn.ConvTranspose2d(100, 512, 4, 1, 0, bias=False),
                                    nn.BatchNorm2d(512),
                                    nn.ReLU(True))

        # input 512*4*4
        self.layer2 = nn.Sequential(nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
                                    nn.BatchNorm2d(256),
                                    nn.ReLU(True),
                                    nn.Dropout2d(0.5))
        # input 256*8*8
        self.layer3 = nn.Sequential(nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
                                    nn.BatchNorm2d(128),
                                    nn.ReLU(True),
                                    nn.Dropout2d(0.5))
        # input 128*16*16
        self.layer4 = nn.Sequential(nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
                                    nn.BatchNorm2d(64),
                                    nn.ReLU(True),
                                    nn.Dropout2d(0.5))
        # input 64*32*32
        self.layer5 = nn.Sequential(nn.ConvTranspose2d(64, 1, 4, 2, 1, bias=False),
                                    nn.Tanh())

        # output 1*64*64

        self.embedding = nn.Embedding(10, 100)

    def forward(self, noise, label):  # noise shape: (,100)

        label_embedding = self.embedding(label)
        x = torch.mul(noise, label_embedding)
        x = x.view(-1, 100, 1, 1)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        return x


class Discriminator(nn.Module):

    def __init__(self):
        super(Discriminator, self).__init__()

        # input 1*64*64
        self.layer1 = nn.Sequential(nn.Conv2d(1, 64, 4, 2, 1, bias=False),
                                    nn.LeakyReLU(0.2, True))

        # input 64*32*32
        self.layer2 = nn.Sequential(nn.Conv2d(64, 128, 4, 2, 1, bias=False),
                                    nn.BatchNorm2d(128),
                                    nn.LeakyReLU(0.2, True),
                                    nn.Dropout2d(0.7))
        # input 128*16*16
        self.layer3 = nn.Sequential(nn.Conv2d(128, 256, 4, 2, 1, bias=False),
                                    nn.BatchNorm2d(256),
                                    nn.LeakyReLU(0.2, True),
                                    nn.Dropout2d(0.6))
        # input 256*8*8
        self.layer4 = nn.Sequential(nn.Conv2d(256, 512, 4, 2, 1, bias=False),
                                    nn.BatchNorm2d(512),
                                    nn.LeakyReLU(0.2, True),
                                    nn.Dropout2d(0.5))
        # input 512*4*4
        self.validity_layer = nn.Sequential(nn.Conv2d(512, 1, 4, 1, 0, bias=False),
                                            nn.Sigmoid())

        self.label_layer = nn.Sequential(nn.Conv2d(512, 10, 4, 1, 0, bias=False),
                                         nn.LogSoftmax(dim=1))

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        validity = self.validity_layer(x)
        plabel = self.label_layer(x)

        validity = validity.view(-1)
        plabel = plabel.view(-1, 10)

        return validity, plabel

In [16]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        m.weight.data.normal_(0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        m.weight.data.normal_(1.0, 0.02)
        m.bias.data.fill_(0)


gen = Generator().to(device)
gen.apply(weights_init)

disc = Discriminator().to(device)
disc.apply(weights_init)

paramsG = list(gen.parameters())
print(len(paramsG))

paramsD = list(disc.parameters())
print(len(paramsD))

optimG = optim.Adam(gen.parameters(), 0.0002, betas=(0.5, 0.999))   
optimD = optim.Adam(disc.parameters(), 0.0002, betas=(0.5, 0.999))


real_labels = 0.7 + 0.5 * torch.rand(7, device=device)   # 0.7 - 1.2
fake_labels = 0.3 * torch.rand(7, device=device)   # 0 - 0.3
epochs = 1

validity_loss = nn.BCELoss()

14
12


In [17]:
trainloader=train_loader2

save_dir="./res1030/dcgan2nsl"

epochs=2

import copy

print(save_dir)    
if not os.path.exists(save_dir): os.makedirs(save_dir)

batch_size=b_s

for epoch in range(1, epochs + 1):

    for idx, (images, labels) in enumerate(trainloader, 0):

        batch_size = images.size(0)
        labels = labels.to(device)
        images = images.to(device)

        real_label = real_labels[idx % 7]
        fake_label = fake_labels[idx % 7]


        if idx % 4 == 0:
            real_label, fake_label = fake_label, real_label

        # ---------------------
        #         disc
        # ---------------------

        optimD.zero_grad()

        # real
        validity_label = torch.full((batch_size,), real_label, device=device)

        pvalidity, plabels = disc(images)

        errD_real_val = validity_loss(pvalidity, validity_label)
        errD_real_label = F.nll_loss(plabels, labels)

        errD_real = errD_real_val + errD_real_label
        errD_real.backward()

        D_x = pvalidity.mean().item()

        # fake
        noise = torch.randn(batch_size, 100, device=device)
        sample_labels = torch.randint(0, 8, (batch_size,), device=device, dtype=torch.long)

        fakes = gen(noise, sample_labels)

        validity_label.fill_(fake_label)

        pvalidity, plabels = disc(fakes.detach())

        errD_fake_val = validity_loss(pvalidity, validity_label)
        errD_fake_label = F.nll_loss(plabels, sample_labels)

        errD_fake = errD_fake_val + errD_fake_label
        errD_fake.backward()

        D_G_z1 = pvalidity.mean().item()

        # finally update the params!
        errD = errD_real + errD_fake

        optimD.step()

        # ------------------------
        #      gen
        # ------------------------

        optimG.zero_grad()

        noise = torch.randn(batch_size, 100, device=device)
        sample_labels = torch.randint(0, 8, (batch_size,), device=device, dtype=torch.long)

        validity_label.fill_(1)

        fakes = gen(noise, sample_labels)
        pvalidity, plabels = disc(fakes)

        errG_val = validity_loss(pvalidity, validity_label)
        errG_label = F.nll_loss(plabels, sample_labels)

        errG = errG_val + errG_label
        errG.backward()

        D_G_z2 = pvalidity.mean().item()

        optimG.step()

        print("[{}/{}] [{}/{}] D_x: [{:.4f}] D_G: [{:.4f}/{:.4f}] G_loss: [{:.4f}] D_loss: [{:.4f}] D_label: [{:.4f}] "
              .format(epoch, epochs, idx, len(trainloader), D_x, D_G_z1, D_G_z2, errG, errD,
                      errD_real_label + errD_fake_label + errG_label))
        
    cur_model_wts = copy.deepcopy(gen.state_dict())
    path_to_save_paramOnly = os.path.join(save_dir, 'dcgan-2nsl-epoch-{}.GNet'.format(epoch))
    torch.save(cur_model_wts, path_to_save_paramOnly)
    
    cur_model_wts = copy.deepcopy(disc.state_dict())
    path_to_save_paramOnly = os.path.join(save_dir, 'dcgan-2nsl-epoch-{}.DNet'.format(epoch))
    torch.save(cur_model_wts, path_to_save_paramOnly)



./res1030/dcgan2nsl
[1/2] [0/13070] D_x: [0.0329] D_G: [0.0088/0.5930] G_loss: [3.0126] D_loss: [13.9591] D_label: [10.4721] 
[1/2] [1/13070] D_x: [0.0193] D_G: [0.5094/0.0106] G_loss: [8.5570] D_loss: [13.1654] D_label: [13.6895] 
[1/2] [2/13070] D_x: [0.3908] D_G: [0.3982/0.7977] G_loss: [1.2613] D_loss: [6.5235] D_label: [6.1158] 
[1/2] [3/13070] D_x: [0.8811] D_G: [0.2830/0.3525] G_loss: [3.3535] D_loss: [13.6112] D_label: [15.2863] 
[1/2] [4/13070] D_x: [0.4174] D_G: [0.0597/0.1731] G_loss: [7.2591] D_loss: [9.2458] D_label: [12.0724] 
[1/2] [5/13070] D_x: [0.7213] D_G: [0.8344/0.2110] G_loss: [6.5972] D_loss: [5.7150] D_label: [8.9642] 
[1/2] [6/13070] D_x: [0.8687] D_G: [0.6398/0.3326] G_loss: [4.3459] D_loss: [3.6632] D_label: [6.2436] 
[1/2] [7/13070] D_x: [0.4722] D_G: [0.3824/0.1631] G_loss: [2.1280] D_loss: [3.3525] D_label: [2.3246] 
[1/2] [8/13070] D_x: [0.0204] D_G: [0.9838/0.1648] G_loss: [6.9308] D_loss: [4.9622] D_label: [8.4329] 
[1/2] [9/13070] D_x: [0.4513] D_G: [0

[1/2] [78/13070] D_x: [0.9937] D_G: [0.0350/0.2713] G_loss: [8.1432] D_loss: [4.1660] D_label: [9.1168] 
[1/2] [79/13070] D_x: [0.8515] D_G: [0.3834/0.3807] G_loss: [2.6172] D_loss: [6.5385] D_label: [7.3990] 
[1/2] [80/13070] D_x: [0.9872] D_G: [0.0709/0.0701] G_loss: [7.5993] D_loss: [9.1929] D_label: [7.8202] 
[1/2] [81/13070] D_x: [0.7975] D_G: [0.0943/0.5967] G_loss: [7.7527] D_loss: [3.2074] D_label: [9.5869] 
[1/2] [82/13070] D_x: [0.7070] D_G: [0.7551/0.0789] G_loss: [8.0039] D_loss: [6.1935] D_label: [10.1764] 
[1/2] [83/13070] D_x: [0.9949] D_G: [0.6985/0.4800] G_loss: [6.7656] D_loss: [2.1289] D_label: [8.1297] 
[1/2] [84/13070] D_x: [0.3908] D_G: [0.0319/0.1447] G_loss: [3.7743] D_loss: [4.9972] D_label: [2.4374] 
[1/2] [85/13070] D_x: [0.4133] D_G: [0.0888/0.0006] G_loss: [12.0950] D_loss: [2.3570] D_label: [5.9276] 
[1/2] [86/13070] D_x: [0.4657] D_G: [0.4933/0.3343] G_loss: [3.4490] D_loss: [1.7670] D_label: [2.6840] 
[1/2] [87/13070] D_x: [0.9827] D_G: [0.6960/0.1213] G

[1/2] [164/13070] D_x: [0.8741] D_G: [0.9686/0.2055] G_loss: [2.5237] D_loss: [4.8435] D_label: [3.9017] 
[1/2] [165/13070] D_x: [0.7454] D_G: [0.1805/0.4702] G_loss: [2.4507] D_loss: [3.8145] D_label: [4.6218] 
[1/2] [166/13070] D_x: [0.9810] D_G: [0.3449/0.4903] G_loss: [3.0001] D_loss: [2.5564] D_label: [5.0634] 
[1/2] [167/13070] D_x: [0.8304] D_G: [0.7085/0.7084] G_loss: [4.0044] D_loss: [4.7988] D_label: [7.5709] 
[1/2] [168/13070] D_x: [0.9660] D_G: [0.2494/0.0542] G_loss: [6.6006] D_loss: [4.9632] D_label: [4.4439] 
[1/2] [169/13070] D_x: [0.9063] D_G: [0.2867/0.7963] G_loss: [5.6760] D_loss: [2.7607] D_label: [7.0034] 
[1/2] [170/13070] D_x: [0.9117] D_G: [0.3005/0.8186] G_loss: [0.2980] D_loss: [4.7944] D_label: [4.2334] 
[1/2] [171/13070] D_x: [0.3423] D_G: [0.5146/0.1752] G_loss: [4.2794] D_loss: [7.2491] D_label: [8.0129] 
[1/2] [172/13070] D_x: [0.8474] D_G: [0.5903/0.0910] G_loss: [9.8449] D_loss: [6.8361] D_label: [11.9196] 
[1/2] [173/13070] D_x: [0.3931] D_G: [0.0374/

[1/2] [248/13070] D_x: [0.7021] D_G: [0.6326/0.0488] G_loss: [7.3007] D_loss: [4.0677] D_label: [6.7957] 
[1/2] [249/13070] D_x: [0.9531] D_G: [0.7584/0.6000] G_loss: [3.9827] D_loss: [4.0835] D_label: [5.4057] 
[1/2] [250/13070] D_x: [0.9345] D_G: [0.3952/0.6599] G_loss: [0.5388] D_loss: [0.4618] D_label: [0.4609] 
[1/2] [251/13070] D_x: [0.3965] D_G: [0.2694/0.9426] G_loss: [1.7407] D_loss: [5.9086] D_label: [5.9981] 
[1/2] [252/13070] D_x: [0.1579] D_G: [0.0159/0.4084] G_loss: [1.3759] D_loss: [6.2163] D_label: [1.5906] 
[1/2] [253/13070] D_x: [0.8240] D_G: [0.2752/0.4635] G_loss: [5.6358] D_loss: [3.6990] D_label: [7.4914] 
[1/2] [254/13070] D_x: [0.6405] D_G: [0.1724/0.1178] G_loss: [4.8257] D_loss: [2.2236] D_label: [4.1402] 
[1/2] [255/13070] D_x: [0.7226] D_G: [0.2227/0.1808] G_loss: [3.1898] D_loss: [5.7819] D_label: [6.4807] 
[1/2] [256/13070] D_x: [0.4117] D_G: [0.9595/0.7337] G_loss: [4.4504] D_loss: [6.3924] D_label: [9.1212] 
[1/2] [257/13070] D_x: [0.7183] D_G: [0.5713/0

[1/2] [333/13070] D_x: [0.9979] D_G: [0.5560/0.7448] G_loss: [2.4440] D_loss: [7.0819] D_label: [6.8492] 
[1/2] [334/13070] D_x: [0.7965] D_G: [0.0027/0.0154] G_loss: [8.5361] D_loss: [6.8678] D_label: [10.6738] 
[1/2] [335/13070] D_x: [0.9909] D_G: [0.0471/0.0387] G_loss: [5.0186] D_loss: [7.0985] D_label: [8.8056] 
[1/2] [336/13070] D_x: [0.9957] D_G: [0.0912/0.1695] G_loss: [8.3100] D_loss: [7.8731] D_label: [7.4349] 
[1/2] [337/13070] D_x: [0.8883] D_G: [0.2138/0.7948] G_loss: [4.8259] D_loss: [3.7774] D_label: [7.2638] 
[1/2] [338/13070] D_x: [0.7665] D_G: [0.5259/0.4399] G_loss: [6.5395] D_loss: [1.4597] D_label: [6.0924] 
[1/2] [339/13070] D_x: [0.9925] D_G: [0.0197/0.7484] G_loss: [3.4247] D_loss: [2.4951] D_label: [4.9292] 
[1/2] [340/13070] D_x: [0.9941] D_G: [0.1070/0.1689] G_loss: [6.1087] D_loss: [9.4290] D_label: [7.3375] 
[1/2] [341/13070] D_x: [0.2090] D_G: [0.4480/0.0388] G_loss: [5.4753] D_loss: [10.5050] D_label: [10.3068] 
[1/2] [342/13070] D_x: [0.8620] D_G: [0.438

...

[2/2] [12684/13070] D_x: [0.7899] D_G: [0.5593/0.4953] G_loss: [0.7027] D_loss: [1.8443] D_label: [0.0000] 
[2/2] [12685/13070] D_x: [0.9047] D_G: [0.2196/0.3750] G_loss: [0.9809] D_loss: [1.1484] D_label: [0.0002] 
[2/2] [12686/13070] D_x: [0.6642] D_G: [0.2578/0.1698] G_loss: [1.7734] D_loss: [0.8192] D_label: [0.0007] 
[2/2] [12687/13070] D_x: [0.6028] D_G: [0.4466/0.2444] G_loss: [1.4091] D_loss: [1.1418] D_label: [0.0042] 
[2/2] [12688/13070] D_x: [0.3218] D_G: [0.4880/0.1283] G_loss: [2.0534] D_loss: [1.1538] D_label: [0.0010] 
[2/2] [12689/13070] D_x: [0.7923] D_G: [0.5680/0.5574] G_loss: [0.5844] D_loss: [0.7996] D_label: [0.0002] 
[2/2] [12690/13070] D_x: [0.7920] D_G: [0.2768/0.4326] G_loss: [0.8379] D_loss: [0.5869] D_label: [0.0001] 
[2/2] [12691/13070] D_x: [0.7703] D_G: [0.2603/0.7555] G_loss: [0.2925] D_loss: [0.6455] D_label: [0.0129] 
[2/2] [12692/13070] D_x: [0.7438] D_G: [0.5247/0.5388] G_loss: [0.6184] D_loss: [1.9512] D_label: [0.0239] 
[2/2] [12693/13070] D_x: [0.

[2/2] [12764/13070] D_x: [0.8998] D_G: [0.6092/0.4236] G_loss: [0.8590] D_loss: [2.4800] D_label: [0.0001] 
[2/2] [12765/13070] D_x: [0.6352] D_G: [0.0728/0.3016] G_loss: [1.1987] D_loss: [0.8743] D_label: [0.0001] 
[2/2] [12766/13070] D_x: [0.5215] D_G: [0.1219/0.3320] G_loss: [1.1030] D_loss: [0.9669] D_label: [0.0089] 
[2/2] [12767/13070] D_x: [0.6346] D_G: [0.3358/0.4268] G_loss: [5.0443] D_loss: [0.9559] D_label: [4.1929] 
[2/2] [12768/13070] D_x: [0.9006] D_G: [0.7769/0.4369] G_loss: [0.8280] D_loss: [1.9745] D_label: [0.0006] 
[2/2] [12769/13070] D_x: [0.6974] D_G: [0.2631/0.2608] G_loss: [1.3441] D_loss: [1.0198] D_label: [0.0003] 
[2/2] [12770/13070] D_x: [0.7692] D_G: [0.5796/0.6485] G_loss: [0.4330] D_loss: [1.1903] D_label: [0.0004] 
[2/2] [12771/13070] D_x: [0.7342] D_G: [0.1418/0.4972] G_loss: [0.6988] D_loss: [0.8153] D_label: [0.0670] 
[2/2] [12772/13070] D_x: [0.7333] D_G: [0.6545/0.0895] G_loss: [2.4136] D_loss: [1.8310] D_label: [0.0024] 
[2/2] [12773/13070] D_x: [0.

[2/2] [12844/13070] D_x: [0.8606] D_G: [0.4286/0.2422] G_loss: [1.4182] D_loss: [2.3614] D_label: [0.0000] 
[2/2] [12845/13070] D_x: [0.7335] D_G: [0.5068/0.6213] G_loss: [0.4772] D_loss: [0.9020] D_label: [0.0021] 
[2/2] [12846/13070] D_x: [0.8377] D_G: [0.3888/0.1968] G_loss: [1.6254] D_loss: [1.2046] D_label: [0.0001] 
[2/2] [12847/13070] D_x: [0.6719] D_G: [0.1959/0.2375] G_loss: [1.4375] D_loss: [0.7527] D_label: [0.0015] 
[2/2] [12848/13070] D_x: [0.8920] D_G: [0.7982/0.2584] G_loss: [1.3551] D_loss: [2.1649] D_label: [0.0020] 
[2/2] [12849/13070] D_x: [0.8073] D_G: [0.4822/0.3345] G_loss: [1.0956] D_loss: [1.2472] D_label: [0.0024] 
[2/2] [12850/13070] D_x: [0.6332] D_G: [0.4213/0.1872] G_loss: [1.6757] D_loss: [0.9349] D_label: [0.0005] 
[2/2] [12851/13070] D_x: [0.7620] D_G: [0.1383/0.2918] G_loss: [1.2317] D_loss: [2.4287] D_label: [1.7050] 
[2/2] [12852/13070] D_x: [0.6698] D_G: [0.3576/0.2270] G_loss: [1.4827] D_loss: [2.0552] D_label: [0.0000] 
[2/2] [12853/13070] D_x: [0.

[2/2] [12924/13070] D_x: [0.8929] D_G: [0.5238/0.4715] G_loss: [0.7519] D_loss: [2.7557] D_label: [0.0000] 
[2/2] [12925/13070] D_x: [0.6437] D_G: [0.2480/0.2835] G_loss: [1.2804] D_loss: [1.0415] D_label: [0.1599] 
[2/2] [12926/13070] D_x: [0.5397] D_G: [0.2838/0.3918] G_loss: [0.9370] D_loss: [1.0652] D_label: [0.0001] 
[2/2] [12927/13070] D_x: [0.5567] D_G: [0.1189/0.3421] G_loss: [1.0733] D_loss: [0.8667] D_label: [0.0009] 
[2/2] [12928/13070] D_x: [0.8567] D_G: [0.3886/0.2246] G_loss: [1.5184] D_loss: [2.4700] D_label: [0.0249] 
[2/2] [12929/13070] D_x: [0.8757] D_G: [0.4778/0.2399] G_loss: [1.4294] D_loss: [0.5884] D_label: [0.0017] 
[2/2] [12930/13070] D_x: [0.5541] D_G: [0.7569/0.2001] G_loss: [2.3063] D_loss: [2.5898] D_label: [1.3340] 
[2/2] [12931/13070] D_x: [0.8010] D_G: [0.2994/0.2626] G_loss: [1.3392] D_loss: [0.7238] D_label: [0.0020] 
[2/2] [12932/13070] D_x: [0.9021] D_G: [0.5154/0.1323] G_loss: [2.0731] D_loss: [2.7133] D_label: [0.1051] 
[2/2] [12933/13070] D_x: [0.

[2/2] [13004/13070] D_x: [0.8486] D_G: [0.1677/0.2257] G_loss: [1.4885] D_loss: [3.7996] D_label: [0.0004] 
[2/2] [13005/13070] D_x: [0.8498] D_G: [0.2666/0.6974] G_loss: [0.3604] D_loss: [0.4463] D_label: [0.0001] 
[2/2] [13006/13070] D_x: [0.7274] D_G: [0.2892/0.4801] G_loss: [0.7337] D_loss: [0.7372] D_label: [0.0000] 
[2/2] [13007/13070] D_x: [0.7848] D_G: [0.2953/0.6561] G_loss: [0.4218] D_loss: [1.0669] D_label: [0.0007] 
[2/2] [13008/13070] D_x: [0.9153] D_G: [0.2299/0.1556] G_loss: [1.8602] D_loss: [4.4109] D_label: [0.7003] 
[2/2] [13009/13070] D_x: [0.9371] D_G: [0.2301/0.1372] G_loss: [1.9862] D_loss: [0.5578] D_label: [0.0001] 
[2/2] [13010/13070] D_x: [0.8625] D_G: [0.5142/0.3966] G_loss: [0.9249] D_loss: [1.3363] D_label: [0.0002] 
[2/2] [13011/13070] D_x: [0.5370] D_G: [0.3078/0.2068] G_loss: [1.5761] D_loss: [1.0467] D_label: [0.0049] 
[2/2] [13012/13070] D_x: [0.8366] D_G: [0.4235/0.0804] G_loss: [2.5217] D_loss: [2.2715] D_label: [0.0021] 
[2/2] [13013/13070] D_x: [0.